In [82]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

In [83]:
# Load a specific sheet
results = pd.read_excel('Concept testing.xlsx', sheet_name='Python2')

overall_results = results[results['Delay'] == 'Overall']

minutes = [3,4,5,6,7,8,9,10,11,12,13,14,15,20,25]
minute_results = {}

for m in minutes:
    minute_results[m] = results[results['Delay'] == f'{m} minute']

In [84]:
# Choose default positions
default_pos = 'top right'
alternate_pos = ['bottom center', 'middle left', 'middle right']

def compute_label_positions(x, y, default_pos='top right', threshold=0.01):
    """
    Assign label positions to avoid overlaps.
    x, y: coordinates of points (Series or array)
    """
    # Convert to numpy arrays to avoid pandas index issues
    x = np.asarray(x)
    y = np.asarray(y)

    n = len(x)
    positions = [default_pos] * n

    for i in range(n):
        for j in range(i):
            dx = abs(x[i] - x[j])
            dy = abs(y[i] - y[j])
            if dx < threshold or dy < threshold:
                positions[i] = alternate_pos[(i + j) % len(alternate_pos)]

    return positions

In [85]:
import numpy as np
import plotly.graph_objects as go

# -------------------------------
# Pareto front function
# -------------------------------
def pareto_front_max(df, x_col, y_col):
    points = df[[x_col, y_col]].values
    is_pareto = np.ones(points.shape[0], dtype=bool)

    for i, point in enumerate(points):
        if is_pareto[i]:
            dominated = np.any(
                (points[:, 0] >= point[0]) &
                (points[:, 1] >= point[1]) &
                ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
            )
            if dominated:
                is_pareto[i] = False

    return df[is_pareto]


# -------------------------------
# Measure definitions
# -------------------------------
measure_names = {
    "M0": "M0: Base",
    "M1": "M1: Cancel 31200",
    "M2": "M2: Priority ICE",
    "M3": "M3: Priority 31200",
    "M4/9": "M4/9: Skip Hengelo",
    "M5/10": "M5/10: Stop Almelo",
    "M6": "M6: Overtake Rijssen",
    "M7": "M7: Short-turn Hengelo",
    "M8": "M8: Alternative path",
    "M11": "M11: Platform Hengelo"
}

# Enforced legend order
measure_order = ['M0','M1','M2','M3','M4/9','M5/10','M6','M7','M8','M11']

# -------------------------------
# Add short + full names safely
# -------------------------------
def add_measure_names(df, id_col):
    df['ShortID'] = df[id_col].astype(str).apply(
        lambda x: x.split(":")[0] if ":" in x else x
    )
    df['Measure_FullName'] = df['ShortID'].map(measure_names)
    return df

# -------------------------------
# Add legend entries IN FIXED ORDER
# -------------------------------
def add_legend_measures(fig):
    for mid in measure_order:
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode='markers',
            marker=dict(size=10, color='skyblue'),
            name=measure_names[mid],
            showlegend=True
        ))

# ===============================
# OVERALL RESULTS
# ===============================
df = overall_results.copy()
df = add_measure_names(df, 'Unnamed: 0')

# Category scores
df['Cat1'] = df[['KPI1: Capacity to maintain path',
                 'KPI2: ICE delay recovery',
                 'KPI 6: Cancelled ICE stops',
                 'KPI 8: Costs']].mean(axis=1)

df['Cat2'] = df[['KPI3: Domestic punctuality',
                 'KPI4: Transfer reliability',
                 'KPI 5: Domestic delay',
                 'KPI 7: Cancelled domestic stops',
                 'KPI 8: Costs']].mean(axis=1)

pareto_df = pareto_front_max(df, 'Cat1', 'Cat2').sort_values('Cat1')

fig = go.Figure()
positions = compute_label_positions(df['Cat1'], df['Cat2'])

# Scatter: measures
fig.add_trace(go.Scatter(
    x=df['Cat1'],
    y=df['Cat2'],
    mode='markers+text',
    text=df['ShortID'],
    textposition=positions,
    cliponaxis=False,
    name='Measures',
    marker=dict(size=10, color='skyblue', opacity=0.7,
                line=dict(width=1, color='black')),
    customdata=df['Measure_FullName'],
    hovertemplate=(
        "<b>%{customdata}</b><br>"
        "Good for ICE: %{x:.2f}<br>"
        "Good for Domestic Services: %{y:.2f}"
        "<extra></extra>"
    )
))

# Pareto front
fig.add_trace(go.Scatter(
    x=pareto_df['Cat1'],
    y=pareto_df['Cat2'],
    mode='lines+markers',
    name='Pareto Front',
    line=dict(color='red', width=3),
    marker=dict(size=14, color='red'),
    hoverinfo='skip'
))

# Legend (fixed order)
add_legend_measures(fig)

fig.update_layout(
    title='Pareto Front Analysis',
    xaxis_title='ICE favouring KPIs',
    yaxis_title='Domestic favouring KPIs',
    template='plotly_white',
    width=900,
    height=650
)

fig.write_image("Pareto_graphs/overall_pareto.png")

# ===============================
# PER-MINUTE RESULTS
# ===============================
for m in minutes:
    df = minute_results[m].copy()
    df = add_measure_names(df, 'Unnamed: 0')

    df['Cat1'] = df[['KPI1: Capacity to maintain path',
                     'KPI2: ICE delay recovery',
                     'KPI 6: Cancelled ICE stops',
                     'KPI 8: Costs']].mean(axis=1)

    df['Cat2'] = df[['KPI3: Domestic punctuality',
                     'KPI4: Transfer reliability',
                     'KPI 5: Domestic delay',
                     'KPI 7: Cancelled domestic stops',
                     'KPI 8: Costs']].mean(axis=1)

    pareto_df = pareto_front_max(df, 'Cat1', 'Cat2').sort_values('Cat1')

    fig = go.Figure()
    positions = compute_label_positions(df['Cat1'], df['Cat2'])

    fig.add_trace(go.Scatter(
        x=df['Cat1'],
        y=df['Cat2'],
        mode='markers+text',
        text=df['ShortID'],
        textposition=positions,
        cliponaxis=False,
        name=f'Measures ({m} min)',
        marker=dict(size=10, color='skyblue', opacity=0.7,
                    line=dict(width=1, color='black')),
        customdata=df['Measure_FullName'],
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "Good for ICE: %{x:.2f}<br>"
            "Good for Domestic Services: %{y:.2f}"
            "<extra></extra>"
        )
    ))

    fig.add_trace(go.Scatter(
        x=pareto_df['Cat1'],
        y=pareto_df['Cat2'],
        mode='lines+markers',
        name=f'Pareto Front ({m} min)',
        line=dict(color='red', width=3),
        marker=dict(size=14, color='red'),
        hoverinfo='skip'
    ))

    # Legend (fixed order)
    add_legend_measures(fig)

    fig.update_layout(
        title=f'Pareto Front Analysis – {m} Minute Delay',
        xaxis_title='ICE favouring KPIs',
        yaxis_title='Domestic favouring KPIs',
        template='plotly_white',
        width=900,
        height=650
    )

    fig.write_image(f"Pareto_graphs/pareto_{m}_minute.png")






In [86]:
import numpy as np
import plotly.graph_objects as go

# --------------------------------------------------
# Pareto front function
# --------------------------------------------------
def pareto_front_max(df, x_col, y_col):
    points = df[[x_col, y_col]].values
    is_pareto = np.ones(points.shape[0], dtype=bool)

    for i, point in enumerate(points):
        if is_pareto[i]:
            dominated = np.any(
                (points[:, 0] >= point[0]) &
                (points[:, 1] >= point[1]) &
                ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
            )
            if dominated:
                is_pareto[i] = False

    return df[is_pareto]

# --------------------------------------------------
# Define measure names
# --------------------------------------------------
measure_names = [
    "Cancel the Arriva stop train 31200 in Hengelo in case of ICE delay.",
    "Grant the ICE priority over the Arriva stop train 31200 at Oldenzaal in case of ICE delay.",
    "Do not grant the ICE priority over the Arriva stop train 31200 when the ICE is delayed.",
    "Remove Hengelo as a stop for the ICE in case of ICE delay",
    "Use the overtaking track at Rijssen in case of ICE delay.",
    "Allow a delayed ICE to short-turn in Hengelo.",
    "Use predefined alternative timetable paths for the ICE.",
    "Remove Hengelo as a stop for the ICE in the timetable.",
    "Reintroduce stopping at Almelo instead of Hengelo.",
    "Construct an additional platform at Hengelo."
]

# --------------------------------------------------
# Precompute frames for all minutes
# --------------------------------------------------
frames = []

for m in minutes:
    df = minute_results[m].copy()
    
    # Compute Cat1 and Cat2
    df['ICE'] = df[['KPI1: Capacity to maintain path',
                 'KPI2: ICE delay recovery',
                 'KPI 6: Cancelled ICE stops',
                 'KPI 8: Costs']].mean(axis=1)
    df['Domestic'] = df[['KPI3: Domestic punctuality',
                 'KPI4: Transfer reliability',
                 'KPI 5: Domestic delay',
                 'KPI 7: Cancelled domestic stops',
                 'KPI 8: Costs']].mean(axis=1)

    # Map measure descriptions
    measure_desc_map = dict(zip([f"M{i}" for i in range(1, 11)], measure_names))
    df['MeasureDesc'] = df['Unnamed: 0'].map(measure_desc_map)

    # Compute label positions
    label_positions = compute_label_positions(df['ICE'].values, df['Domestic'].values)

    # Pareto front
    pareto_df = pareto_front_max(df, 'ICE', 'Domestic').sort_values('ICE')

    frames.append(
        go.Frame(
            name=str(m),
            data=[
                # All designs
                go.Scatter(
                    x=df['ICE'],
                    y=df['Domestic'],
                    mode='markers+text',
                    text=df['Unnamed: 0'],  # M1, M2, etc.
                    textposition=label_positions,
                    cliponaxis=False,
                    marker=dict(size=10, color='skyblue', opacity=0.7, line=dict(width=1, color='black')),
                    name='Designs',
                    hovertemplate=(
                        "<b>%{text}</b><br>" +
                        "%{customdata}<br>" +
                        "ICE: %{x:.2f}<br>" +
                        "Domestic: %{y:.2f}<extra></extra>"
                    ),
                    customdata=df['MeasureDesc']
                ),
                # Pareto front
                go.Scatter(
                    x=pareto_df['ICE'],
                    y=pareto_df['Domestic'],
                    mode='lines+markers',
                    line=dict(color='red', width=3, smoothing=1.3),
                    marker=dict(symbol='circle', size=14, color='red', line=dict(width=2, color='darkred')),
                    name='Pareto Front',
                    text=pareto_df['Unnamed: 0'],
                    hovertemplate=(
                        "<b>%{text}</b><br>" +
                        "%{customdata}<br>" +
                        "ICE: %{x:.2f}<br>" +
                        "Domestic: %{y:.2f}<extra></extra>"
                    ),
                    customdata=pareto_df['MeasureDesc']
                )
            ],
            layout=go.Layout(title_text=f'Pareto Front Analysis – {m} Minute Delay')
        )
    )

# --------------------------------------------------
# Initial figure (first minute)
# --------------------------------------------------
init_df = minute_results[minutes[0]].copy()
init_df['ICE'] = init_df[['KPI3: Domestic punctuality',
                 'KPI4: Transfer reliability',
                 'KPI 5: Domestic delay',
                 'KPI 7: Cancelled domestic stops',
                 'KPI 8: Costs']].mean(axis=1)
init_df['Domestic'] = init_df[['KPI3: Domestic punctuality',
                 'KPI4: Transfer reliability',
                 'KPI 5: Domestic delay',
                 'KPI 7: Cancelled domestic stops',
                 'KPI 8: Costs']].mean(axis=1)

measure_desc_map = dict(zip([f"M{i}" for i in range(1, 11)], measure_names))
init_df['MeasureDesc'] = init_df['Unnamed: 0'].map(measure_desc_map)

init_positions = compute_label_positions(init_df['ICE'].values, init_df['Domestic'].values)
init_pareto = pareto_front_max(init_df, 'ICE', 'Domestic').sort_values('ICE')

fig = go.Figure(
    data=[
        go.Scatter(
            x=init_df['ICE'],
            y=init_df['Domestic'],
            mode='markers+text',
            text=init_df['Unnamed: 0'],  # Only M1–M10
            textposition=init_positions,
            cliponaxis=False,
            name='Designs',
            marker=dict(size=10, color='skyblue', opacity=0.7, line=dict(width=1, color='black')),
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "%{customdata}<br>" +
                "ICE: %{x:.2f}<br>" +
                "Domestic: %{y:.2f}<extra></extra>"
            ),
            customdata=init_df['MeasureDesc']
        ),
        go.Scatter(
            x=init_pareto['ICE'],
            y=init_pareto['Domestic'],
            mode='lines+markers',
            name='Pareto Front',
            line=dict(color='red', width=3, smoothing=1.3),
            marker=dict(symbol='circle', size=14, color='red', line=dict(width=2, color='darkred')),
            text=init_pareto['Unnamed: 0'],
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "%{customdata}<br>" +
                "ICE: %{x:.2f}<br>" +
                "Domestic: %{y:.2f}<extra></extra>"
            ),
            customdata=init_pareto['MeasureDesc']
        )
    ],
    frames=frames
)

# --------------------------------------------------
# Slider + layout
# --------------------------------------------------
fig.update_layout(
    template='plotly_white',
    width=900,
    height=650,
    xaxis_title='ICE favouring KPIs',
    yaxis_title='Domestic favouring KPIs',
    title=f'Pareto Front Analysis – {minutes[0]} Minute Delay',
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "Delay: "},
        "pad": {"t": 50},
        "steps": [
            {
                "method": "animate",
                "label": f"{m} min",
                "args": [
                    [str(m)],
                    {"mode": "immediate", "frame": {"duration": 600}, "transition": {"duration": 300}}
                ]
            }
            for m in minutes
        ]
    }]
)

fig.write_html("pareto_slider.html")
fig.show()


In [ ]:
# import pandas as pd
# import plotly.graph_objects as go

# # --------------------------------------------------
# # Configuration
# # --------------------------------------------------
# KPI_COLUMNS = ['KPI1: Capacity to maintain path',
#                'KPI2: ICE delay recovery', 
#                'KPI3: Domestic punctuality', 
#                'KPI4: Transfer reliability', 
#                'KPI 5: Domestic delay', 
#                'KPI 6: Cancelled ICE stops',
#                'KPI 7: Cancelled domestic stops', 
#                'KPI 8: Costs']

# measure_order = ['M0','M1', 'M2', 'M3', 'M4/9', 'M5/10', 'M6', 'M7', 'M8', 'M11']

# # --------------------------------------------------
# # Build dataset: average KPI per measure per minute
# # --------------------------------------------------
# records = []

# for m in minutes:
#     df = minute_results[m].copy()
#     df['KPI_AVG'] = df[KPI_COLUMNS].mean(axis=1)
    
#     for _, row in df.iterrows():
#         records.append({
#             'Minute': m,
#             'Measure': row['Unnamed: 0'],
#             'KPI_AVG': row['KPI_AVG']
#         })

# kpi_avg_df = pd.DataFrame(records)

# # Keep only measures that exist, in desired order
# measure_order = [m for m in measure_order if m in kpi_avg_df['Measure'].unique()]

# # --------------------------------------------------
# # Create plot
# # --------------------------------------------------
# fig = go.Figure()

# for measure in measure_order:
#     df_m = kpi_avg_df[kpi_avg_df['Measure'] == measure]
    
#     fig.add_trace(
#         go.Scatter(
#             x=df_m['Minute'],
#             y=df_m['KPI_AVG'],
#             mode='lines+markers',
#             name=measure
#         )
#     )

# # --------------------------------------------------
# # Layout
# # --------------------------------------------------
# fig.update_layout(
#     template='plotly_white',
#     width=1000,
#     height=650,
#     title='Average KPI Value per Measure over Delay Minutes',
#     xaxis_title='Delay (minutes)',
#     yaxis_title='Average KPI Value',
#     legend_title='Measure'
# )

# # --------------------------------------------------
# # Save as PNG (requires kaleido)
# # --------------------------------------------------
# fig.write_image("average_kpi_per_measure_over_minutes.png", scale=2)

# fig.show()



In [91]:
# import pandas as pd
# import plotly.graph_objects as go

# # --------------------------------------------------
# # Configuration
# # --------------------------------------------------
# KPI_COLUMNS = ['KPI1','KPI2','KPI3','KPI4','KPI5','KPI6','KPI7','KPI8']

# # Desired logical order
# measure_order = ['M0','M1', 'M2', 'M3', 'M4/9', 'M5/10', 'M6', 'M7', 'M8', 'M11']

# # --------------------------------------------------
# # Build KPI average dataset
# # --------------------------------------------------
# records = []

# for m in minutes:
#     df = minute_results[m].copy()
#     df['KPI_AVG'] = df[KPI_COLUMNS].mean(axis=1)
    
#     for _, row in df.iterrows():
#         records.append({
#             'Minute': m,
#             'Measure': row['Unnamed: 0'],  # M1–M10
#             'KPI_AVG': row['KPI_AVG']
#         })

# kpi_avg_df = pd.DataFrame(records)

# # Keep only measures that exist in the data, in the given order
# measure_order = [m for m in measure_order if m in kpi_avg_df['Measure'].unique()]

# # --------------------------------------------------
# # Create figure and traces
# # --------------------------------------------------
# fig = go.Figure()

# for measure in measure_order:
#     df_m = kpi_avg_df[kpi_avg_df['Measure'] == measure]
    
#     fig.add_trace(
#         go.Scatter(
#             x=df_m['Minute'],
#             y=df_m['KPI_AVG'],
#             mode='lines+markers',
#             name=measure,
#             visible=False
#         )
#     )

# # Show first measure by default
# fig.data[0].visible = True

# # --------------------------------------------------
# # Slider
# # --------------------------------------------------
# slider_steps = []

# for i, measure in enumerate(measure_order):
#     slider_steps.append({
#         "method": "update",
#         "label": measure,
#         "args": [
#             {"visible": [j == i for j in range(len(measure_order))]},
#             {"title": f"Average KPI Value over Time – {measure}"}
#         ]
#     })

# # --------------------------------------------------
# # Layout
# # --------------------------------------------------
# fig.update_layout(
#     template='plotly_white',
#     width=900,
#     height=600,
#     title=f"Average KPI Value over Time – {measure_order[0]}",
#     xaxis_title="Delay (minutes)",
#     yaxis_title="Average KPI Value",
#     sliders=[{
#         "active": 0,
#         "pad": {"t": 50},
#         "steps": slider_steps
#     }]
# )

# fig.write_html("kpi_avg_slider_per_measure_custom_order.html")
# fig.show()


In [89]:
import pandas as pd

results = pd.read_excel('Concept testing.xlsx', sheet_name='Python2')
df = results[results['Delay'] == 'Overall'].copy()

kpi_cols = [col for col in df.columns if col.startswith("KPI")]
id_col = [col for col in df.columns if col not in kpi_cols and col != 'Delay'][0]

df[kpi_cols] = df[kpi_cols].apply(pd.to_numeric)

# --------------------------------------------------
# Dominance helpers
# --------------------------------------------------
def dominates(a, b, kpi_cols):
    ge_all = (a[kpi_cols] >= b[kpi_cols]).all()
    gt_any = (a[kpi_cols] > b[kpi_cols]).any()
    return ge_all and gt_any

# --------------------------------------------------
# Build dominance map
# --------------------------------------------------
dominates_map = {row[id_col]: [] for _, row in df.iterrows()}
dominated_by_any = set()

for _, a in df.iterrows():
    for _, b in df.iterrows():
        if a[id_col] == b[id_col]:
            continue
        if dominates(a, b, kpi_cols):
            dominates_map[a[id_col]].append(b[id_col])
            dominated_by_any.add(b[id_col])

# --------------------------------------------------
# Non-dominated measures
# --------------------------------------------------
non_dominated = [
    m for m in dominates_map.keys()
    if m not in dominated_by_any
]

# --------------------------------------------------
# Print results
# --------------------------------------------------
print("=== Non-dominated measures and who they dominate (Overall) ===\n")
for m in non_dominated:
    print(f"{m} dominates: {dominates_map[m]}")
print("\n")

for m in minutes:
    df = minute_results[m].copy()

    kpi_cols = [col for col in df.columns if col.startswith("KPI")]
    id_col = [col for col in df.columns if col not in kpi_cols and col != 'Delay'][0]

    df[kpi_cols] = df[kpi_cols].apply(pd.to_numeric)

    def dominates(a, b, kpi_cols):
        ge_all = (a[kpi_cols] >= b[kpi_cols]).all()
        gt_any = (a[kpi_cols] > b[kpi_cols]).any()
        return ge_all and gt_any

    dominates_map = {row[id_col]: [] for _, row in df.iterrows()}
    dominated_by_any = set()

    for _, a in df.iterrows():
        for _, b in df.iterrows():
            if a[id_col] == b[id_col]:
                continue
            if dominates(a, b, kpi_cols):
                dominates_map[a[id_col]].append(b[id_col])
                dominated_by_any.add(b[id_col])

    non_dominated = [
        m_id for m_id in dominates_map.keys()
        if m_id not in dominated_by_any
    ]

    print(f"=== Non-dominated measures and who they dominate for {m} minutes ===\n")
    for m_id in non_dominated:
        print(f"{m_id} dominates: {dominates_map[m_id]}")
    print("\n")




=== Non-dominated measures and who they dominate (Overall) ===

M0: Base dominates: ['M3: Priority 31200']
M1: Cancel 31200 dominates: []
M2: Priority ICE dominates: []
M4/9: Skip Hengelo dominates: []
M11: Platform Hengelo dominates: []
M6: Overtake Rijssen dominates: []
M8: Alternative path dominates: ['M7: Short-turn Hengelo']
M5/10: Stop Almelo dominates: []


=== Non-dominated measures and who they dominate for 3 minutes ===

M0: Base dominates: ['M1: Cancel 31200', 'M3: Priority 31200', 'M6: Overtake Rijssen', 'M5/10: Stop Almelo']
M2: Priority ICE dominates: ['M1: Cancel 31200', 'M3: Priority 31200', 'M6: Overtake Rijssen', 'M5/10: Stop Almelo']
M4/9: Skip Hengelo dominates: ['M7: Short-turn Hengelo']
M11: Platform Hengelo dominates: []
M8: Alternative path dominates: ['M7: Short-turn Hengelo']


=== Non-dominated measures and who they dominate for 4 minutes ===

M0: Base dominates: ['M1: Cancel 31200', 'M3: Priority 31200', 'M6: Overtake Rijssen', 'M5/10: Stop Almelo']
M2: Prio

In [90]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from itertools import combinations
import re
import os

# -------------------------------
# Pareto front function
# -------------------------------
def pareto_front_max(df, x_col, y_col):
    points = df[[x_col, y_col]].values
    is_pareto = np.ones(points.shape[0], dtype=bool)

    for i, point in enumerate(points):
        if is_pareto[i]:
            dominated = np.any(
                (points[:, 0] >= point[0]) &
                (points[:, 1] >= point[1]) &
                ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
            )
            if dominated:
                is_pareto[i] = False

    return df[is_pareto]

# -------------------------------
# Label positions function
# -------------------------------
def compute_label_positions(x, y):
    return ['top center'] * len(x)

# -------------------------------
# Measure names and order
# -------------------------------
measure_names = {
    "M0": "M0: Base",
    "M1": "M1: Cancel 31200",
    "M2": "M2: Priority ICE",
    "M3": "M3: Priority 31200",
    "M4/9": "M4/9: Skip Hengelo",
    "M5/10": "M5/10: Stop Almelo",
    "M6": "M6: Overtake Rijssen",
    "M7": "M7: Short-turn Hengelo",
    "M8": "M8: Alternative path",
    "M11": "M11: Platform Hengelo"
}

measure_order = ['M0','M1','M2','M3','M4/9','M5/10','M6','M7','M8','M11']

# -------------------------------
# Add measure names and short IDs
# -------------------------------
def add_measure_names(df, id_col):
    if id_col in df.columns:
        df['ShortID'] = df[id_col].apply(
            lambda x: str(x).split(":")[0] if ":" in str(x) else str(x)
        )
        df['Measure_FullName'] = df['ShortID'].map(measure_names).fillna(df['ShortID'])
    else:
        df['ShortID'] = df.index.astype(str)
        df['Measure_FullName'] = df['ShortID']
    return df

# -------------------------------
# Add legend entries in fixed order
# -------------------------------
def add_legend_measures(fig, df, measure_order):
    df_lookup = df.set_index('ShortID')

    for mid in measure_order:
        if mid in df_lookup.index:
            fig.add_trace(go.Scatter(
                x=[None],
                y=[None],
                mode='markers',
                marker=dict(size=10, color='skyblue'),
                name=df_lookup.loc[mid, 'Measure_FullName'],
                showlegend=True
            ))

# -------------------------------
# Sanitize filenames (Windows-safe)
# -------------------------------
def safe_filename(name):
    return re.sub(r'[<>:"/\\|?*]', '_', name)

# -------------------------------
# Prepare DataFrame
# -------------------------------
df = overall_results.copy()
df = add_measure_names(df, 'Unnamed: 0')  # adjust if needed

# KPI columns
kpi_columns = [col for col in df.columns if col.startswith("KPI")]

# Output folder
output_folder = "Pareto_graphs2"
os.makedirs(output_folder, exist_ok=True)

# -------------------------------
# Loop over all KPI pairs
# -------------------------------
for x_col, y_col in combinations(kpi_columns, 2):

    pareto_df = pareto_front_max(df, x_col, y_col).sort_values(x_col)

    fig = go.Figure()
    positions = compute_label_positions(df[x_col].values, df[y_col].values)

    # All measures
    fig.add_trace(go.Scatter(
        x=df[x_col],
        y=df[y_col],
        mode='markers+text',
        text=df['ShortID'],
        textposition=positions,
        cliponaxis=False,
        name='Measures',
        marker=dict(
            size=10,
            color='skyblue',
            opacity=0.7,
            line=dict(width=1, color='black')
        ),
        customdata=df['Measure_FullName'],
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            f"{x_col}: "+"%{x:.2f}<br>"
            f"{y_col}: "+"%{y:.2f}"
            "<extra></extra>"
        )
    ))

    # Pareto front
    fig.add_trace(go.Scatter(
        x=pareto_df[x_col],
        y=pareto_df[y_col],
        mode='lines+markers',
        name='Pareto Front',
        line=dict(color='red', width=3, smoothing=1.3),
        marker=dict(
            symbol='circle',
            size=14,
            color='red',
            line=dict(width=2, color='darkred')
        ),
        hoverinfo='skip'
    ))

    # Ordered legend
    add_legend_measures(fig, df, measure_order)

    fig.update_layout(
        title=dict(
            text=f'Pareto Front: {x_col} vs {y_col}',
            x=0.5,
            font=dict(size=16)
        ),
        xaxis_title=x_col,
        yaxis_title=y_col,
        template='plotly_white',
        width=900,
        height=650
    )

    #fig.show()

    filename = os.path.join(
        output_folder,
        f"{safe_filename(x_col)}_vs_{safe_filename(y_col)}_pareto.png"
    )
    fig.write_image(filename)
